### oliveyoung review web crawler - korea(평점 낮은순)

## 📋 크롤러 개요

이 노트북은 **Selenium(undetected_chromedriver)** 을 이용해 올리브영(oliveyoung.co.kr) 특정 상품의 리뷰 페이지에 접속한 뒤, 화면에 노출되는 리뷰 개수 제한을 우회하여 리뷰 원문 데이터를 수집하는 크롤러입니다.

### 🔍 수집 방식
1. 브라우저를 직접 실행하여 올리브영 로그인 진행 (본인 계정 필요, 자동 로그인 아님)
2. 로그인 이후 브라우저가 정상적으로 발급받는 인증 토큰(JWT)을 가로채어 확보
3. 확보한 토큰으로 올리브영 리뷰 API(`/review/api/v2/reviews/cursor`)를 **커서 기반(cursor-based) 방식**으로 반복 호출하며 페이지네이션 처리
4. 수집한 리뷰를 리스트에 누적한 뒤 pandas DataFrame으로 정리, CSV로 저장

### 📦 수집 데이터 항목
| 컬럼명 | 설명 |
|---|---|
| goods_no | 상품 코드 |
| writer | 작성자 닉네임 |
| skin_type | 피부 타입 (A01-A07) |
| skin_tone | 피부 톤 (B01-B06) |
| skin_trouble | 피부 고민 (C01-C13) |
| rating | 평점 |
| review_date | 작성일 |
| review_txt | 리뷰 본문 |
| helpful_cnt | 도움돼요 수 |

### ⚙️ 정렬 기준 (예시 설정)
현재 API 요청 파라미터의 `sortType`은 `'RATING_ASC'`(**평점 낮은순**)으로 설정되어 있습니다. 이는 부정 리뷰를 우선적으로 다량 확보하기 위한 **예시 설정**이며, 목적에 따라 아래처럼 변경할 수 있습니다.
- `'RATING_ASC'` — 평점 낮은순 (현재 설정)
- `'RATING_DESC'` — 평점 높은순
- `'DATETIME_DESC'` — 최신순

값을 바꾸고 싶다면 리뷰 수집 셀(step 6) 안의 `sortType` 파라미터를 직접 수정하면 됩니다.

### ✅ 실행 전 준비사항
- **상품 링크(URL)를 미리 확보해두어야 합니다.** 올리브영 상품 상세페이지 URL 형식은 다음과 같습니다:
  ```
  https://www.oliveyoung.co.kr/store/goods/getGoodsDetail.do?goodsNo=A000000000000
  ```
  `goodsNo` 값이 상품별 고유 코드이며, 아래 step 3 셀 실행 시 URL을 직접 입력받아 자동으로 파싱합니다.
- 로컬에 설치된 **Chrome 브라우저 버전을 미리 확인**해두어야 합니다 (`VERSION_MAIN` 값에 반영, `chrome://settings/help`에서 확인 가능).
- 올리브영 계정으로 **직접 로그인하는 단계가 포함**되어 있어, 완전 자동화가 아닌 반자동(semi-automatic) 크롤러입니다.


In [ ]:
#step 1. 필요한 라이브러리 로딩
from selenium import webdriver
import undetected_chromedriver as uc
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import WebDriverException
from selenium.webdriver.common.action_chains import ActionChains
import urllib.request, urllib
import pandas as pd
from bs4 import BeautifulSoup
import os, tempfile, time, math, random, re
from datetime import datetime, timedelta
import unicodedata
import json

In [ ]:
# 기본 저장 경로: 현재 작업 폴더 하위 output/kr_asc/ (엔터 시 자동 적용, 원하는 경로 직접 입력 가능)
#step 2. 파일 저장경로 설정

#1. 파일 저장경로
web_name = 'oliveyoung'

f_dir = input('1. 파일 저장 경로를 입력하세요.')
if f_dir =='':
    f_dir = os.path.join(os.getcwd(), 'output', 'kr_asc') + os.sep
print('1. 파일 저장 경로 : %s'%f_dir)
print('='*80)

n = time.localtime()
s = '%04d-%02d-%02d-%02d-%02d-%02d' % (n.tm_year, n.tm_mon, n.tm_mday, n.tm_hour, n.tm_min, n.tm_sec)

os.makedirs(f_dir+s+'-'+web_name)  #폴더생성   - make directory
os.chdir(f_dir+s+'-'+web_name) #현재 작업 폴더로 지정 - change directory

ff_name = f_dir+s+'-'+web_name+'\\'+s+'-'+web_name+'.txt'
fc_name = f_dir+s+'-'+web_name+'\\'+s+'-'+web_name+'.csv'
#fx_name = f_dir+s+'-'+web_name+'\\'+s+'-'+web_name+'.xlsx'

In [ ]:
# 수집 대상 상품 링크 예시 (goodsNo 파라미터로 상품을 식별함)
# https://www.oliveyoung.co.kr/store/goods/getGoodsDetail.do?goodsNo=A000000000000
#
# ※ 실제 상품 링크는 아래 step 3 셀 실행 시 input()으로 직접 입력받습니다.
#    (여러 상품을 수집할 경우, 상품마다 이 노트북을 재실행하거나 goodsNo만 바꿔가며 반복 실행)


In [ ]:
import subprocess

# 실행 중인 chrome 프로세스 강제 종료
subprocess.run(['taskkill', '/f', '/im', 'chrome.exe'], capture_output=True)
subprocess.run(['taskkill', '/f', '/im', 'chromedriver.exe'], capture_output=True)

import time
time.sleep(2)

# step 1. 드라이버 설정
CHROME_PATH = r"C:\Program Files\Google\Chrome\Application\chrome.exe"
VERSION_MAIN = 147  # 본인 로컬 Chrome 버전에 맞게 수정 필요 (chrome://settings/help 에서 확인)

options = Options()
#options.add_argument('--headless=new')

user_data = os.path.join(tempfile.gettempdir(), 'us_profile_clean')
os.makedirs(user_data, exist_ok=True)

driver = uc.Chrome(
    options=options,
    browser_executable_path=CHROME_PATH,
    version_main=VERSION_MAIN,
    user_data_dir=user_data,
    use_subprocess=True,
    patcher_force_close=True,
    debug=True,
)

# step 2. 로그인
driver.get('https://www.oliveyoung.co.kr/store/login/loginForm.do')
input('로그인 완료 후 엔터를 눌러주세요...')

In [ ]:
# 위 안내대로 미리 확보한 상품 링크(URL)를 여기서 입력받습니다.
# step 3. 상품 링크 입력 및 접속
url = input('고객 후기를 수집할 제품의 링크를 입력하세요.')
print('후기 수집할 제품 링크 :', url) 
print('='*80)

# goodsNo 파싱
from urllib.parse import urlparse, parse_qs
parsed = urlparse(url)
goods_no = parse_qs(parsed.query).get('goodsNo', [None])[0]
print(f"goodsNo : {goods_no}")
print('='*80)

# 상품 페이지 접속
driver.get(url)
time.sleep(2)
driver.maximize_window()
time.sleep(3)

# step 4. 리뷰 섹션 이동 및 최신순 정렬
review_btn = driver.find_element(By.CSS_SELECTOR, 'button[class*="ReviewArea_btn-review"]')
driver.execute_script("arguments[0].click();", review_btn)
time.sleep(1)
print('===리뷰 섹션으로 이동합니다.===')

driver.execute_script("""
    function clickInShadow(root, depth) {
        if (depth > 5) return false;
        const buttons = [...root.querySelectorAll('button.pc-sort-button')];
        for (let btn of buttons) {
            if (btn.textContent.trim() === '평점 낮은순') {
                btn.click();
                return true;
            }
        }
        for (let el of [...root.querySelectorAll('*')]) {
            if (el.shadowRoot) {
                if (clickInShadow(el.shadowRoot, depth + 1)) return true;
            }
        }
        return false;
    }
    
    const host = document.querySelector('oy-review-review-in-product');
    const result = clickInShadow(host.shadowRoot, 0);
    return result;
""")
time.sleep(2)
print('===평점 낮은순으로 정렬 완료===')

In [ ]:
# 화면에 노출되는 전체 리뷰 개수를 먼저 파악한 뒤, 실제 수집할 건수를 입력받습니다.
#step 5. 전체 리뷰수 추출 및 수집할 리뷰 건수 입력받기

#전체 리뷰수
html = driver.page_source
soup = BeautifulSoup(html, 'html.parser')

total = soup.select('span[class*="GoodsDetailTabs_count"]')[0]
total1 = total.text.replace(',', '')
total2 = re.search(r'\d+', total1)
total_cnt = int(total2.group())

#cnt, page_cnt
cnt_input = input('3. 수집할 댓글의 수를 입력해주세요 : ')
if cnt_input == '' or int(cnt_input) > total_cnt:
    cnt = total_cnt
else:
    cnt = int(cnt_input)

print('3. 전체 리뷰 %s건 중 수집할 건수는 %s건 입니다.'%(total_cnt, cnt))
print('='*80)

#브랜드명
brand_1 = soup.select_one('button[class*="TopUtils_btn-brand"]')
brand = brand_1.text


### 평점 낮은순으로 요청 건수만큼 수집

In [ ]:
# ※ 본인 올리브영 계정으로 로그인한 뒤, 브라우저가 정상적으로 발급받는 인증 토큰을
# 학술 연구 목적의 1회성 데이터 수집에 한해 활용함 (토큰 값 자체는 저장/공유하지 않음)

# =============================================
# step 5. auth_token 캡처
# =============================================

# 1. 인터셉터 심기
driver.execute_script("""
    window._captured_auth = null;
    const originalSetHeader = XMLHttpRequest.prototype.setRequestHeader;
    XMLHttpRequest.prototype.setRequestHeader = function(header, value) {
        if (header === 'Authorization') {
            window._captured_auth = value;
        }
        return originalSetHeader.apply(this, arguments);
    };
    const originalFetch = window.fetch;
    window.fetch = function(...args) {
        const req = args[1];
        if (req && req.headers && req.headers['Authorization']) {
            window._captured_auth = req.headers['Authorization'];
        }
        return originalFetch.apply(this, args);
    };
""")

# 2. 브라우저에서 직접 스크롤해서 리뷰 로드 유도
input('브라우저에서 리뷰 섹션까지 스크롤 후 엔터를 눌러주세요...')

# 3. 토큰 캡처
auth_token = driver.execute_script("return window._captured_auth;")
print('auth_token 캡처 성공!' if auth_token else 'auth_token 캡처 실패') 

In [ ]:
# 수집 결과를 담을 리스트 초기화 + 커서 기반 페이지네이션에 필요한 변수 선언
# (next_cursor_id / cursor_count / cursor_score: 다음 페이지 요청 시 필요한 커서 값)
count = 0
page = 0
seen_url = set()

index2 = []
goods_no2 = []
writer2 = []
skin_type2 = []
skin_tone2 = []
skin_trouble2 = []
rating2 = []
review_date2 = []
review_txt2 = []
helpful_cnt2 = []

next_cursor_id = None
next_cursor_count = None
next_cursor_score = None

In [ ]:
# ↓ sortType 파라미터를 여기서 확인/수정하세요 (현재: 'RATING_ASC' = 평점 낮은순, 예시 설정)
# =============================================
# step 6. 리뷰 수집
# =============================================
ff_name = f'oliveyoung_{goods_no}_reviews.txt'

f = open(ff_name, 'a', encoding='UTF-8')

while count < cnt:
    print('===%s 페이지 조회중입니다.==='%(page+1))

    result = driver.execute_script("""
        const response = await fetch('https://m.oliveyoung.co.kr/review/api/v2/reviews/cursor', {
            method: 'POST',
            headers: {
                'Content-Type': 'application/json',
                'Accept': 'application/json, text/plain, */*',
                'Origin': 'https://www.oliveyoung.co.kr',
                'Referer': 'https://www.oliveyoung.co.kr/',
                'Authorization': arguments[4]
            },
            body: JSON.stringify({
                goodsNumber: arguments[0],
                size: 10,
                sortType: 'RATING_ASC',
                reviewType: 'ALL',
                cursorId: arguments[1],
                cursorCount: arguments[2],
                cursorScore: arguments[3]
            })
        });
        return await response.json();
    """, goods_no, next_cursor_id, next_cursor_count, next_cursor_score, auth_token)

    # result 체크
    if not result:
        print('result가 None입니다. 종료합니다.')
        break

    data = result.get('data')
    if not data:
        print('data가 없습니다. result:', result)
        break

    reviews = data.get('goodsReviewList', [])
    has_next = data.get('hasNext', False)
    login_required = data.get('loginRequired', False)

    if login_required:
        print('토큰 만료! 브라우저에서 리뷰 섹션 스크롤 후 엔터를 눌러주세요...')
        input()
        auth_token = driver.execute_script("return window._captured_auth;")
        print('auth_token 재캡처 완료!')
        continue

    if not reviews:
        print('더 이상 리뷰가 없습니다.')
        print('loginRequired:', login_required)
        print('has_next:', has_next)
        print('result 전체:', result)

        break

    for r in reviews:
        if count >= cnt:
            break

        review_id = str(r['reviewId'])
        review_txt = r['content']

        if review_id in seen_url or review_txt in seen_url:
            continue

        seen_url.add(review_id)
        seen_url.add(review_txt)

        count += 1

        writer       = r['profileDto']['memberNickname']
        skin_type    = r['profileDto'].get('skinType', []) or None
        skin_tone    = r['profileDto'].get('skinTone', []) or None
        skin_trouble = r['profileDto'].get('skinTrouble', []) or None
        rating       = r['reviewScore']
        review_date  = r['createdDateTime'][:10]
        helpful_cnt  = r['recommendCount']

        print(f'{count}. {writer} | 별점:{rating} | {review_date}')

        index2.append(count)
        goods_no2.append(goods_no)
        writer2.append(writer)
        skin_type2.append(skin_type)
        skin_tone2.append(skin_tone)
        skin_trouble2.append(skin_trouble)
        rating2.append(rating)
        review_date2.append(review_date)
        review_txt2.append(review_txt)
        helpful_cnt2.append(helpful_cnt)

        f.write(f'[{count}] 작성자:{writer} | 별점:{rating} | 작성일:{review_date}\n')
        f.write(f'피부타입:{skin_type} | 피부톤:{skin_tone} | 피부고민:{skin_trouble}\n')
        f.write(f'리뷰:{review_txt}\n')
        f.write(f'추천수:{helpful_cnt}\n\n')

    next_cursor_id = data.get('nextCursorId')
    next_cursor_count = data.get('nextCursorCount')
    next_cursor_score = data.get('nextCursorScore')

    if not has_next:
        print('마지막 페이지입니다.')
        break

    page += 1
    driver.execute_script("window.scrollBy(0, 1000);")
    time.sleep(2)

f.close()
print(f'총 {count}개 수집 완료')
print('='*80)

In [ ]:
# 수집한 리뷰 리스트를 DataFrame으로 정리 후 CSV로 저장
#step 6. csv, xlsx 형태로 저장하기
oliveyoung = pd.DataFrame()
oliveyoung['goods_no'] = pd.Series(goods_no2)
oliveyoung['writer'] = pd.Series(writer2)
oliveyoung['skin_type'] = pd.Series(skin_type2)
oliveyoung['skin_tone'] = pd.Series(skin_tone2)
oliveyoung['skin_trouble'] = pd.Series(skin_trouble2)
oliveyoung['rating'] = pd.Series(rating2)
oliveyoung['review_date'] = pd.Series(review_date2)
oliveyoung['review_txt'] = pd.Series(review_txt2)
oliveyoung['helpful_cnt'] = pd.Series(helpful_cnt2)

oliveyoung.to_csv(fc_name, encoding='utf-8-sig', index=False)
#oliveyoung.to_excel(fx_name, index=False, engine='openpyxl')

print('\n')
print('='*100)
print('1. 파일 저장 완료 txt : %s'%ff_name)
print('2. 파일 저장 완료 csv : %s'%fc_name)
#print('3. 파일 저장 완료 xlsx : %s'%fx_name)
print('='*100)

In [ ]:
# 실험용 테스트 셀 - 최종 크롤링 로직에는 미사용 (참고용으로만 남김)
#new - API호출 방식
result = driver.execute_script("""
    const response = await fetch('https://m.oliveyoung.co.kr/review/api/v2/reviews/cursor', {
        method: 'POST',
        headers: {
            'Content-Type': 'application/json',
            'Accept': 'application/json, text/plain, */*',
            'Origin': 'https://www.oliveyoung.co.kr',
            'Referer': 'https://www.oliveyoung.co.kr/'
        },
        body: JSON.stringify({
            goodsNumber: arguments[0],
            size: 10,
            sortType: 'DATETIME_DESC',
            reviewType: 'ALL',
            cursorId: null,
            cursorCount: null,
            cursorScore: null
        })
    });
    return await response.json();
""", goods_no)

print(result)